In [1]:
import pandas as pd
import requests
import pymysql
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4120\3910071212.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
current_date = datetime.now()
formatted_date = current_date.strftime("%Y%m%d")
date=timedelta(1)+datetime.now()
print(formatted_date,date)

20240702 2024-07-03 12:22:29.792475


In [4]:
def quater(date):
    if date.month>=1 and date.month<=3:
      return {'pyt_fetch':str((date-relativedelta(years=1)).year)+'1001T000000','db_fetch_start':str((date-relativedelta(years=1)).year)+'0701','db_fetch_end':str((date-relativedelta(years=1)).year)+'0930'}
    elif date.month>=4 and date.month<=6:
      return {'pyt_fetch':str(date.year)+'0101T000000','db_fetch_start':str((date-relativedelta(years=1)).year)+'1001','db_fetch_end':str((date-relativedelta(years=1)).year)+'1231'}
    elif date.month>=7 and date.month<=9:
      return {'pyt_fetch':str(date.year)+'0401T000000','db_fetch_start':str(date.year)+'0101','db_fetch_end':str(date.year)+'0331'}
    else:
      return {'pyt_fetch':str(date.year)+'0701T000000','db_fetch_start':str(date.year)+'0401','db_fetch_end':str(date.year)+'0630'}

quat=quater(current_date)   
print(quat)


{'pyt_fetch': '20240401T000000', 'db_fetch_start': '20240101', 'db_fetch_end': '20240331'}


In [5]:

# Your credentials
username = 'MondelezPortland'
password = 'BakerySLX360'

# URL and parameters
url = "https://saas75.shoplogix.com/web/api/export/summary"
params = {
    'start': quat['db_fetch_start'],
    'end': quat['db_fetch_end'],
    'metrics': 'total,scrap,jobs',
    'machines': '862C993B-C83D-1AE6-4191-A278302D7B59,C7CC9488-34B8-4654-35D6-6D4B073DB900,398F6011-E432-5D96-AE00-BC601454EFC9,58E4DF72-A0BA-9FCC-1B3E-4E1B362C29FD,6922AF33-1EAF-6C10-F8BB-5E4350895477,E734E668-C538-05D8-48E4-A27AAD11969A,1C0A3C71-CAAA-894B-CD18-71FC05CB55B2,8A40D3AD-50DA-6409-051B-5E8BF0C0AFEA,56BD02A6-D1D0-6CA7-58E9-0E595CDDE9CE,93B6A077-27F4-0EBE-BD99-0E595CE36496,8D4D54E7-CE42-589C-52B4-4F0910520C83',
    'groupBy': 'Machine,shiftInstance'
}

# Making the GET request with Basic Auth
try:
    response = requests.get(url, params=params, auth=(username, password))
    response.raise_for_status()  # Raises stored HTTPError, if one occurred
    data=response.json()
    # Output the JSON response
    print(response.json())
except requests.exceptions.HTTPError as errh:
    print(f"HTTP Error: {errh}")
except requests.exceptions.ConnectionError as errc:
    print(f"Error Connecting: {errc}")
except requests.exceptions.Timeout as errt:
    print(f"Timeout Error: {errt}")
except requests.exceptions.RequestException as err:
    print(f"OOps: Something Else: {err}")


{'result': {'query': 'https://saas75.shoplogix.com:443/web/api/export/summary?start=20240101&end=20240331&metrics=total%2Cscrap%2Cjobs&machines=862C993B-C83D-1AE6-4191-A278302D7B59%2CC7CC9488-34B8-4654-35D6-6D4B073DB900%2C398F6011-E432-5D96-AE00-BC601454EFC9%2C58E4DF72-A0BA-9FCC-1B3E-4E1B362C29FD%2C6922AF33-1EAF-6C10-F8BB-5E4350895477%2CE734E668-C538-05D8-48E4-A27AAD11969A%2C1C0A3C71-CAAA-894B-CD18-71FC05CB55B2%2C8A40D3AD-50DA-6409-051B-5E8BF0C0AFEA%2C56BD02A6-D1D0-6CA7-58E9-0E595CDDE9CE%2C93B6A077-27F4-0EBE-BD99-0E595CE36496%2C8D4D54E7-CE42-589C-52B4-4F0910520C83&groupBy=Machine%2CshiftInstance', 'machines': [{'machineId': '862C993B-C83D-1AE6-4191-A278302D7B59', 'machineName': 'L18-Dough Machine 1 - BN', 'erpCode': '', 'shiftInstances': [{'shiftName': 'Shift 3', 'shiftInstance': '862C993B-C83D-1AE6-4191-A278302D7B59 Shift 3 2023-12-31 23:15:00', 'start': '20240101T000000.000', 'end': '20240101T071500.000', 'metrics': {'total': 0.0, 'scrap': 0.0, 'jobs': [{'name': '440000311100', 'inst

In [6]:

flattened_data = []

# Function to flatten data
def flatten_data(machine):
    machine_id = machine['machineId']
    machine_name = machine['machineName']

    for instance in machine['shiftInstances']:
        shift_name = instance['shiftName']
        start = instance['start']
        end = instance['end']

        metrics = instance['metrics']
        total = metrics['total']
        scrap = metrics['scrap']

        # Iterate through jobs if available
        if 'jobs' in metrics:
            for job in metrics['jobs']:
                current_job = job['currentJob']
                scheduled_duration = job['scheduledDuration']
                job_max_run_rate = job['jobMaxRunRate']
                operation_offset = job['operationOffset']
                sku = job['filters']['filter1'] if 'filter1' in job['filters'] else None  # Check if filter1 exists

                # Create a dictionary of the required information
                job_details = {
                    'machineId': machine_id,
                    'machineName': machine_name,
                    'shiftName': shift_name,
                    'start': start,
                    'end': end,
                    'total': total,
                    'scrap': scrap,
                    'currentJob': current_job,
                    'scheduledDuration': scheduled_duration,
                    'jobMaxRunRate': job_max_run_rate,
                    'operationOffset': operation_offset,
                    'sku': sku
                }
                # Append the dictionary to the list
                flattened_data.append(job_details)

# Assuming 'data' is the JSON object loaded from your previous API call
for machine in data['result']['machines']:
    flatten_data(machine)

# Convert the list of dictionaries to a DataFrame
df = pd.DataFrame(flattened_data)
df


,machineId,machineName,shiftName,start,end,total,scrap,currentJob,scheduledDuration,jobMaxRunRate,operationOffset,sku
0,862C993B-C83D-1AE6-4191-A278302D7B59,L18-Dough Machine 1 - BN,Shift 3,20240101T000000.000,20240101T071500.000,0.0,0.0,True,0.000000,690768.000000,-1.0,13.7Z RITZ CRACKERS ORIGINAL 12
1,862C993B-C83D-1AE6-4191-A278302D7B59,L18-Dough Machine 1 - BN,Shift 1,20240101T071500.000,20240101T151500.000,0.0,0.0,True,0.000000,690768.000000,-1.0,13.7Z RITZ CRACKERS ORIGINAL 12
2,862C993B-C83D-1AE6-4191-A278302D7B59,L18-Dough Machine 1 - BN,Shift 2,20240101T151500.000,20240101T231500.000,0.0,0.0,True,0.000000,690768.000000,-1.0,13.7Z RITZ CRACKERS ORIGINAL 12
3,862C993B-C83D-1AE6-4191-A278302D7B59,L18-Dough Machine 1 - BN,Shift 3,20240101T231500.000,20240102T071500.000,80013.0,0.0,True,0.115116,690768.000000,-1.0,13.7Z RITZ CRACKERS ORIGINAL 12
4,862C993B-C83D-1AE6-4191-A278302D7B59,L18-Dough Machine 1 - BN,Shift 1,20240102T071500.000,20240102T151500.000,2168629.0,1008522.0,True,4.213019,690768.000000,-1.0,13.7Z RITZ CRACKERS ORIGINAL 12
...,...,...,...,...,...,...,...,...,...,...,...,...
3017,8D4D54E7-CE42-589C-52B4-4F0910520C83,L05-Case Sealer,Shift 2,20240329T151500.000,20240329T231500.000,0.0,0.0,True,0.000000,383.333333,-1.0,18.2Z CA! COOKIE ORIG 12
3018,8D4D54E7-CE42-589C-52B4-4F0910520C83,L05-Case Sealer,Shift 3,20240329T231500.000,20240330T071500.000,0.0,0.0,True,0.000000,383.333333,-1.0,18.2Z CA! COOKIE ORIG 12
3019,8D4D54E7-CE42-589C-52B4-4F0910520C83,L05-Case Sealer,Shift 1,20240330T071500.000,20240330T151500.000,0.0,0.0,True,0.000000,383.333333,-1.0,18.2Z CA! COOKIE ORIG 12
3020,8D4D54E7-CE42-589C-52B4-4F0910520C83,L05-Case Sealer,Shift 2,20240330T151500.000,20240330T231500.000,0.0,0.0,True,0.000000,383.333333,-1.0,18.2Z CA! COOKIE ORIG 12


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3022 entries, 0 to 3021
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   machineId          3022 non-null   object 
 1   machineName        3022 non-null   object 
 2   shiftName          3022 non-null   object 
 3   start              3022 non-null   object 
 4   end                3022 non-null   object 
 5   total              3022 non-null   float64
 6   scrap              3022 non-null   float64
 7   currentJob         3022 non-null   bool   
 8   scheduledDuration  3022 non-null   float64
 9   jobMaxRunRate      3022 non-null   float64
 10  operationOffset    3022 non-null   float64
 11  sku                3022 non-null   object 
dtypes: bool(1), float64(5), object(6)
memory usage: 262.8+ KB


In [8]:
df['total']=pd.to_numeric(df['total'],errors='coerce')
df['scrap']=pd.to_numeric(df['scrap'],errors='coerce')
df['scheduledDuration']=pd.to_numeric(df['scheduledDuration'],errors='coerce')
df['jobMaxRunRate']=pd.to_numeric(df['jobMaxRunRate'],errors='coerce')
df['operationOffset']=pd.to_numeric(df['operationOffset'],errors='coerce')

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3022 entries, 0 to 3021
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   machineId          3022 non-null   object 
 1   machineName        3022 non-null   object 
 2   shiftName          3022 non-null   object 
 3   start              3022 non-null   object 
 4   end                3022 non-null   object 
 5   total              3022 non-null   float64
 6   scrap              3022 non-null   float64
 7   currentJob         3022 non-null   bool   
 8   scheduledDuration  3022 non-null   float64
 9   jobMaxRunRate      3022 non-null   float64
 10  operationOffset    3022 non-null   float64
 11  sku                3022 non-null   object 
dtypes: bool(1), float64(5), object(6)
memory usage: 262.8+ KB


In [10]:
connection=pymysql.connect(
    host="s13.hosterpk.com",
    user="digita87_muddassir",
    password="Developers000$$$",
    database="digita87_mondeleez"
)
data_tuples=list(df.itertuples(index=False, name=None))
insert_query="""INSERT INTO production (machineId,machineName,shiftName,start,end,total,scrap,currentJob,scheduledDuration,jobMaxRunRate,operationOffset,sku) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)"""

with connection.cursor() as cursor:
    cursor.executemany(insert_query,data_tuples)
    connection.commit()
connection.close()